# Feature-set test: is the manual 36-set's edge real, or selection bias?

**Goal:** you want the single most accurate model for ground-type prediction, and you have a
hand-probed 36-feature XGBoost set that beat forward/backward/full-48 on grouped CV. Hand-tuning
is fine for picking a deployable winner — **but** because the 36 were chosen by looking at the
grouped-CV score, that score is optimistic (selection bias): you searched until something scored
well *on these folds/these 3-4 users*. This notebook quantifies how much of the 36-set's advantage
survives an honest test.

Three numbers, from weakest to strongest evidence:
1. **In-sample grouped CV** — full-48 vs your-36, per fold. (This is the possibly-inflated number.)
2. **Per-fold spread** — is the 36-vs-48 gap larger than the fold-to-fold noise? If not, it's not real.
3. **Nested selection baseline** — let an automatic selector pick features *inside each fold* (no
   leakage) and see what grouped CV it gets. If your manual-36 beats the leakage-free nested score,
   the manual advantage is likely at least partly selection bias (the nested number is the honest
   'what generalises' figure); if manual-36 ≈ nested, your set is trustworthy.

Read together, these tell you whether to trust the 36-set's headline number for deployment.


In [1]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# ================= CONFIG =================
DATA_DIR = '..'            # folder with the *_dry.csv files
# N_SPLITS auto-set after loading (leave-user-out): min #users appearing across all classes
RANDOM_STATE = 42

# person-confound features to always exclude (match your pipeline)
DROP = {'total_p_mean','total_pmax','impact_mag','impact_mag_norm','heel_impulse',
        'heel_mean_ratio','fore_mean_ratio','heel_pmax_ratio','fore_pmax_ratio'} \
       | {f'pressure_{i:02d}_mean_ratio' for i in range(1,13)}
META = {'foot','step_id','surface','source_user','source_recording'}

# >>> PASTE YOUR MANUAL 36-FEATURE SET HERE <<<
MANUAL_36 = ['orient_rest_a_roll3_std', 'gyro_x_std', 'accel_jerk_ratio', 'orient_rest_b_roll3_std', 'accel_spec_centroid', 'accel_hf_frac', 'accel_spec_entropy', 'gyro_hf_frac', 'heel_ripple_frac', 'gyro_x_strike', 'heel_impulse_norm', 'gu_spatial_entropy_stance', 'accel_zcr', 'heel_fore_ratio', 'gu_stance_frac', 'gu_push_peak', 'gyro_z_std', 'gyro_y_strike', 'impact_mag_roll3_std', 'heel_kurt', 'gu_fore_early_frac', 'accel_mag_strike', 'orient_range_a', 'gu_dip_depth', 'gu_press_jitter', 'gu_load_slope', 'loading_rate_norm', 'gu_cop_wander', 'gu_participation_ff', 'orient_range_b', 'gyro_z_strike', 'stance_duration_roll3_std', 'slope_asym', 'fall_slope', 'gyro_y_std', 'gu_fore_heel_toff']

def load_3class():
    dfs=[]
    for fn,lab in [('grass_dry.csv','grass'),('loose_dry.csv','loose'),('paved_dry.csv','paved')]:
        d=pd.read_csv(f'{DATA_DIR}/{fn}'); d['surface']=lab; dfs.append(d)
    df=pd.concat(dfs,ignore_index=True)
    feats=[c for c in df.columns if c not in META and c not in DROP]
    return df, feats

df, FEATS48 = load_3class()
print(f'{len(df)} steps | {len(FEATS48)} candidate features | users {sorted(df.source_user.unique())}')
# leave-user-out folds = the smallest number of distinct users any class has
_users_per_class = df.groupby('surface')['source_user'].nunique()
N_SPLITS = int(_users_per_class.min())
print(f'users per class: {_users_per_class.to_dict()}  ->  N_SPLITS = {N_SPLITS} '
      f'(capped by the class with fewest users; a fold cannot exceed group count)')
if N_SPLITS < 2: raise ValueError('Need >=2 users in every class for grouped CV.')
if MANUAL_36:
    missing=[f for f in MANUAL_36 if f not in df.columns]
    print(f'MANUAL_36: {len(MANUAL_36)} features'+(f'  [WARNING missing from data: {missing}]' if missing else '  (all present)'))
else:
    print('MANUAL_36 is EMPTY — paste your 36-feature list into the config cell to run the comparison.')


15664 steps | 49 candidate features | users ['Catherine', 'Kristian', 'Sam', 'Samantha']
users per class: {'grass': 4, 'loose': 3, 'paved': 4}  ->  N_SPLITS = 3 (capped by the class with fewest users; a fold cannot exceed group count)
MANUAL_36: 36 features  (all present)


In [2]:
# shared XGBoost + grouped-CV harness (identical settings for every feature set tested)
classes = sorted(df.surface.unique())
y   = df['surface'].values
grp = df['source_user'].values
_le = LabelEncoder().fit(classes)

def make_xgb():
    return XGBClassifier(max_depth=4, n_estimators=250, learning_rate=0.05,
                         subsample=0.9, colsample_bytree=0.8, eval_metric='mlogloss',
                         random_state=RANDOM_STATE, verbosity=0)

def grouped_cv(feat_cols, per_fold=True):
    X = df[feat_cols].fillna(0).values
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.empty(len(y), object); fold_mf1=[]; fold_acc=[]
    for tr,te in splitter.split(X,y,grp):
        if len(np.unique(y[tr]))<len(classes): continue
        m=make_xgb(); m.fit(X[tr],_le.transform(y[tr]))
        pred=_le.inverse_transform(m.predict(X[te])); oof[te]=pred
        fold_mf1.append(f1_score(y[te],pred,labels=classes,average='macro'))
        fold_acc.append(accuracy_score(y[te],pred))
    filled=oof!=None
    return dict(acc=accuracy_score(y[filled],oof[filled].astype(str)),
                mf1=f1_score(y[filled],oof[filled].astype(str),labels=classes,average='macro'),
                per_fold_mf1=np.array(fold_mf1), per_fold_acc=np.array(fold_acc))


## 1 & 2. In-sample grouped CV + per-fold spread: full-48 vs your manual-36

In [3]:
r48 = grouped_cv(FEATS48)
print(f'full-48 : Acc {r48["acc"]*100:.1f}%  MacroF1 {r48["mf1"]*100:.1f}%')
print(f'          per-fold MacroF1: {np.round(r48["per_fold_mf1"]*100,1)}  (sd {r48["per_fold_mf1"].std()*100:.1f})')

if MANUAL_36:
    r36 = grouped_cv(MANUAL_36)
    print(f'manual-36: Acc {r36["acc"]*100:.1f}%  MacroF1 {r36["mf1"]*100:.1f}%')
    print(f'          per-fold MacroF1: {np.round(r36["per_fold_mf1"]*100,1)}  (sd {r36["per_fold_mf1"].std()*100:.1f})')
    gap = (r36['mf1']-r48['mf1'])*100
    noise = max(r48['per_fold_mf1'].std(), r36['per_fold_mf1'].std())*100
    print(f'\n36 - 48 MacroF1 gap: {gap:+.1f} pts   |   fold-to-fold sd: ~{noise:.1f} pts')
    if abs(gap) < noise:
        print('READ: the 36-set edge is SMALLER than fold noise -> not distinguishable from chance on this data.')
    else:
        print('READ: the 36-set edge EXCEEDS fold noise -> a real improvement on these folds (still see #3 for leakage).')
else:
    print('\n(paste MANUAL_36 to compare)')


full-48 : Acc 70.9%  MacroF1 46.6%
          per-fold MacroF1: [45.9 46.7 46.5]  (sd 0.3)
manual-36: Acc 71.0%  MacroF1 46.6%
          per-fold MacroF1: [46.7 45.6 47.9]  (sd 0.9)

36 - 48 MacroF1 gap: +0.0 pts   |   fold-to-fold sd: ~0.9 pts
READ: the 36-set edge is SMALLER than fold noise -> not distinguishable from chance on this data.


## 3. Leakage-free baseline: nested forward selection (features chosen INSIDE each fold)

This is the honest 'what generalises' number. Features are selected using only the training users
of each outer fold, so the score has no selection bias. If your manual-36 beats this, some of its
advantage is selection bias (you tuned to these folds); if manual-36 ≈ nested, your set is solid.


In [4]:
def nested_forward_cv(candidate_feats, max_k=15, tol=0.002):
    '''Outer grouped CV; inside each fold, forward-select (on inner grouped CV of the training
    users only) up to max_k features, then score on the held-out outer fold. No leakage.'''
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    Xall = df[candidate_feats].fillna(0).values
    import time as _time
    outer_mf1=[]; picked_sizes=[]
    _folds=[(tr,te) for tr,te in splitter.split(Xall,y,grp) if len(np.unique(y[tr]))==len(classes)]
    print(f'nested forward selection: {len(_folds)} outer folds, max_k={max_k} ...')
    for _fi,(tr,te) in enumerate(_folds,1):
        _t0=_time.time()
        held_out_user = np.unique(grp[te])
        print(f'  fold {_fi}/{len(_folds)} (holding out user {held_out_user}) — selecting...', flush=True)
        ytr, gtr = y[tr], grp[tr]
        inner = StratifiedGroupKFold(n_splits=max(2,min(N_SPLITS-1,len(np.unique(gtr)))), shuffle=True, random_state=0)
        def inner_score(cols):
            sc=[]
            for itr,ite in inner.split(Xall[tr][:,cols],ytr,gtr):
                if len(np.unique(ytr[itr]))<len(classes): continue
                m=make_xgb(); m.fit(Xall[tr][:,cols][itr],_le.transform(ytr[itr]))
                sc.append(f1_score(ytr[ite],_le.inverse_transform(m.predict(Xall[tr][:,cols][ite])),
                                   labels=classes,average='macro'))
            return np.mean(sc) if sc else -1
        chosen=[]; remaining=list(range(len(candidate_feats))); best=-1
        while remaining and len(chosen)<max_k:
            trial=sorted(((inner_score(chosen+[c]),c) for c in remaining),reverse=True)
            if trial[0][0]-best<tol: break
            chosen.append(trial[0][1]); best=trial[0][0]; remaining.remove(trial[0][1])
        m=make_xgb(); m.fit(Xall[tr][:,chosen],_le.transform(ytr))
        pf=f1_score(y[te],_le.inverse_transform(m.predict(Xall[te][:,chosen])),labels=classes,average='macro')
        outer_mf1.append(pf); picked_sizes.append(len(chosen))
        print(f'    -> {len(chosen)} features chosen, held-out MacroF1 {pf*100:.1f}%  ({_time.time()-_t0:.0f}s)', flush=True)
    outer_mf1=np.array(outer_mf1)
    print(f'nested forward: MacroF1 {outer_mf1.mean()*100:.1f}% (sd {outer_mf1.std()*100:.1f})  '
          f'| features chosen per fold: {picked_sizes}')
    return outer_mf1

# NOTE: this is the slow cell (nested loop). It is the leakage-free reference the manual-36 must beat
# to claim a *real* (not selection-biased) advantage.
nested = nested_forward_cv(FEATS48, max_k=15)   # ~5 min on a fast CPU; prints per-fold progress
if MANUAL_36:
    print(f'\nmanual-36 grouped MacroF1: {r36["mf1"]*100:.1f}%   vs   nested (leakage-free): {nested.mean()*100:.1f}%')
    d=(r36['mf1']-nested.mean())*100
    if d > max(nested.std(),r36['per_fold_mf1'].std())*100:
        print(f'   manual-36 is {d:+.1f} pts ABOVE the leakage-free baseline -> that gap is likely SELECTION BIAS,')
        print('   i.e. the manual-36 CV score is optimistic; expect real-world accuracy closer to the nested number.')
    else:
        print(f'   manual-36 is within noise of the leakage-free baseline -> the 36-set is trustworthy, not overfit.')


nested forward selection: 3 outer folds, max_k=15 ...
  fold 1/3 (holding out user ['Sam']) — selecting...
    -> 4 features chosen, held-out MacroF1 47.2%  (68s)
  fold 2/3 (holding out user ['Kristian']) — selecting...
    -> 2 features chosen, held-out MacroF1 46.6%  (44s)
  fold 3/3 (holding out user ['Catherine' 'Samantha']) — selecting...
    -> 5 features chosen, held-out MacroF1 39.7%  (101s)
nested forward: MacroF1 44.5% (sd 3.4)  | features chosen per fold: [4, 2, 5]

manual-36 grouped MacroF1: 46.6%   vs   nested (leakage-free): 44.5%
   manual-36 is within noise of the leakage-free baseline -> the 36-set is trustworthy, not overfit.


## How to read this
- If **manual-36 ≈ full-48** (gap < fold sd): dimensionality reduction didn't really help accuracy;
  prefer the simpler/faster set but don't claim an accuracy win.
- If **manual-36 > full-48 AND manual-36 ≈ nested**: your set is a genuine, trustworthy improvement — use it.
- If **manual-36 > full-48 BUT manual-36 > nested by more than the noise**: the extra edge is selection
  bias. Deploy the set if you like, but report the **nested** number as the honest expected accuracy,
  not the in-sample manual-36 number.

Honest limit: with 3-4 users every number here has large variance. Treat gaps under ~2-3 MacroF1 points
as noise regardless of which way they fall.
